# Sign Language Translator - Video Inference UI
### Dual-Stream SLR - 5-Sign ASL - COMP 5405 Group 11

Upload a single `.mp4` video clip and the model will:
1. Extract RGB frames and MediaPipe keypoints from the clip
2. Run the Dual-Stream (RGB + Keypoint) model
3. Return the predicted ASL word **or** `Sorry, I can't translate this word.`

> **Before running:** set `CHECKPOINT_PATH` in Step 2 to point to your saved checkpoint.


## Step 1 - Install Dependencies

In [42]:
# Uncomment the first block to install all the dependencies
#!pip install -q opencv-python-headless einops mediapipe ipywidgets tqdm
#import IPython; IPython.display.clear_output()
#print('All dependencies installed.')


## Step 2 - Imports & Configuration

Edit `CHECKPOINT_PATH` to match your saved model file.

In [43]:
import json, math, random, warnings, os, tempfile, urllib.request
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision.models as models
import torchvision.transforms as T
import cv2
import mediapipe as mp
from mediapipe.tasks import python as mp_python
from mediapipe.tasks.python import vision as mp_vision
from einops import rearrange
from pathlib import Path
from IPython.display import display, HTML, clear_output
import ipywidgets as widgets
import matplotlib.pyplot as plt

warnings.filterwarnings('ignore')

# CONFIGURATION 
CHECKPOINT_PATH      = './checkpoints/best_10signs.pt'
CONFIDENCE_THRESHOLD = 0.90   
TOP_K                = 1      

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device     : {DEVICE}')
print(f'Checkpoint : {CHECKPOINT_PATH}')


Device     : cpu
Checkpoint : ./checkpoints/best_10signs.pt


## Step 3 - Model Architecture

Re-defines the same architecture used during training so the checkpoint loads correctly.

In [44]:
class RGBStream(nn.Module):
    def __init__(self, d_model, vit_layers, vit_heads, max_frames):
        super().__init__()
        resnet = models.resnet50(weights=models.ResNet50_Weights.IMAGENET1K_V1)
        for name, p in resnet.named_parameters():
            if not any(x in name for x in ['layer3', 'layer4', 'fc']):
                p.requires_grad = False
        self.backbone = nn.Sequential(*list(resnet.children())[:-1])
        self.proj     = nn.Linear(2048, d_model)
        self.pos_emb  = nn.Parameter(torch.randn(1, max_frames, d_model) * 0.02)
        enc = nn.TransformerEncoderLayer(d_model, vit_heads, d_model * 2, 0.1,
                                         batch_first=True, norm_first=True)
        self.vit  = nn.TransformerEncoder(enc, vit_layers)
        self.norm = nn.LayerNorm(d_model)

    def forward(self, rgb, mask=None):
        B, T = rgb.shape[:2]
        x = rearrange(rgb, 'b t c h w -> (b t) c h w')
        x = self.backbone(x).flatten(1)
        x = self.proj(x)
        x = rearrange(x, '(b t) d -> b t d', b=B, t=T)
        x = x + self.pos_emb[:, :T]
        kpm = (mask == 0) if mask is not None else None
        return self.norm(self.vit(x, src_key_padding_mask=kpm))


class STGCNBlock(nn.Module):
    def __init__(self, in_ch, out_ch, t_kernel=5):
        super().__init__()
        self.gcn  = nn.Conv1d(in_ch, out_ch, 1)
        self.tcn  = nn.Sequential(
            nn.BatchNorm1d(out_ch), nn.ReLU(),
            nn.Conv1d(out_ch, out_ch, t_kernel, padding=t_kernel // 2),
            nn.BatchNorm1d(out_ch),
        )
        self.skip = nn.Conv1d(in_ch, out_ch, 1) if in_ch != out_ch else nn.Identity()
        self.relu = nn.ReLU()

    def forward(self, x):
        return self.relu(self.tcn(self.gcn(x)) + self.skip(x))


class KeypointStream(nn.Module):
    def __init__(self, d_model, channels):
        super().__init__()
        layers, in_ch = [], 3
        for out_ch in channels:
            layers.append(STGCNBlock(in_ch, out_ch))
            in_ch = out_ch
        self.stgcn  = nn.Sequential(*layers)
        self.bilstm = nn.LSTM(channels[-1], d_model // 2, num_layers=2,
                              batch_first=True, bidirectional=True, dropout=0.1)
        self.norm = nn.LayerNorm(d_model)

    def forward(self, kp):
        B, T, N, C = kp.shape
        x = rearrange(kp, 'b t n c -> (b n) c t')
        x = self.stgcn(x)
        x = rearrange(x, '(b n) c t -> b n c t', b=B, n=N).mean(1)
        x = rearrange(x, 'b c t -> b t c')
        out, _ = self.bilstm(x)
        return self.norm(out)


class DSFF(nn.Module):
    def __init__(self, d_model, nhead, out_dim):
        super().__init__()
        self.ca_A  = nn.MultiheadAttention(d_model, nhead, 0.1, batch_first=True)
        self.ca_B  = nn.MultiheadAttention(d_model, nhead, 0.1, batch_first=True)
        self.proj  = nn.Linear(2 * d_model, out_dim)
        self.normA = nn.LayerNorm(d_model)
        self.normB = nn.LayerNorm(d_model)
        self.norm  = nn.LayerNorm(out_dim)

    def forward(self, A, B, key_mask=None):
        At, _ = self.ca_A(A, B, B, key_padding_mask=key_mask)
        Bt, _ = self.ca_B(B, A, A, key_padding_mask=key_mask)
        At = self.normA(A + At)
        Bt = self.normB(B + Bt)
        return self.norm(self.proj(torch.cat([At, Bt], dim=-1)))


class TBT(nn.Module):
    def __init__(self, d_model, n_layers, nhead):
        super().__init__()
        enc = nn.TransformerEncoderLayer(d_model, nhead, d_model, 0.1,
                                         batch_first=True, norm_first=True)
        self.enc  = nn.TransformerEncoder(enc, n_layers)
        self.head = nn.Sequential(nn.Linear(d_model, 64), nn.ReLU(),
                                   nn.Linear(64, 1), nn.Sigmoid())

    def forward(self, x, mask=None):
        kpm = (mask == 0) if mask is not None else None
        z   = self.enc(x, src_key_padding_mask=kpm)
        p_t = self.head(z).squeeze(-1)
        return p_t, z


class ClassHead(nn.Module):
    def __init__(self, d_model, n_classes):
        super().__init__()
        self.mlp = nn.Sequential(
            nn.Linear(d_model, d_model), nn.GELU(), nn.Dropout(0.3),
            nn.Linear(d_model, n_classes)
        )

    def forward(self, z, mask=None):
        if mask is not None:
            me = mask.unsqueeze(-1)
            pooled = (z * me).sum(1) / me.sum(1).clamp(min=1)
        else:
            pooled = z.mean(1)
        return self.mlp(pooled)


class DualStreamSLR(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        d  = cfg['d_model']
        dd = cfg['dec_hidden']
        self.stream_a = RGBStream(d, cfg['vit_layers'], cfg['vit_heads'], cfg['max_frames'])
        self.stream_b = KeypointStream(d, cfg['stgcn_channels'])
        self.dsff     = DSFF(d, 4, dd)
        self.tbt      = TBT(dd, cfg['tbt_layers'], cfg['tbt_heads'])
        self.head     = ClassHead(dd, cfg['n_classes'])
        self.lambda_b = cfg['lambda_tbt']

    def forward(self, rgb, kp, mask=None):
        Z_A      = self.stream_a(rgb, mask)
        Z_B      = self.stream_b(kp)
        km       = (mask == 0) if mask is not None else None
        Z_f      = self.dsff(Z_A, Z_B, km)
        p_t, Z_e = self.tbt(Z_f, mask)
        logits   = self.head(Z_e, mask)
        return logits, p_t


print('Model architecture defined.')


Model architecture defined.


## Step 4 - Load Checkpoint

In [45]:
def load_model(ckpt_path, device):
    ckpt_path = Path(ckpt_path)
    assert ckpt_path.exists(), f'Checkpoint not found: {ckpt_path}'
    ckpt      = torch.load(ckpt_path, map_location=device)
    cfg       = ckpt['cfg']
    class2idx = ckpt['class2idx']
    idx2class = ckpt['idx2class']
    model = DualStreamSLR(cfg).to(device)
    model.load_state_dict(ckpt['state_dict'])
    model.eval()
    return model, cfg, class2idx, idx2class

model, CFG, CLASS2IDX, IDX2CLASS = load_model(CHECKPOINT_PATH, DEVICE)
N_CLASSES   = len(CLASS2IDX)
CLASS_NAMES = [IDX2CLASS[i] for i in range(N_CLASSES)]
_meta = torch.load(CHECKPOINT_PATH, map_location='cpu')
print(f'Model loaded  |  {N_CLASSES} classes: {CLASS_NAMES}')
print(f'Checkpoint epoch : {_meta["epoch"]}')
print(f'Best val acc     : {_meta["best_acc"]*100:.1f}%')


Model loaded  |  5 classes: ['bye', 'can', 'help', 'no', 'yes']
Checkpoint epoch : 8
Best val acc     : 90.8%


## Step 5 - Load MediaPipe Keypoint Extractor

In [46]:
MEDIAPIPE_MODELS = {
    'pose': ('pose_landmarker_full.task',
             'https://storage.googleapis.com/mediapipe-models/pose_landmarker/pose_landmarker_full/float16/latest/pose_landmarker_full.task'),
    'hand': ('hand_landmarker.task',
             'https://storage.googleapis.com/mediapipe-models/hand_landmarker/hand_landmarker/float16/latest/hand_landmarker.task'),
}

for key, (fname, url) in MEDIAPIPE_MODELS.items():
    if not Path(fname).exists():
        print(f'Downloading {fname} ...')
        urllib.request.urlretrieve(url, fname)
        print('  Done.')
    else:
        print(f'  {fname} already present.')


class KeypointExtractor:
    BODY  = 33
    FACE  = 468
    HAND  = 21
    TOTAL = 33 + 468 + 21 + 21

    def __init__(self):
        pose_opts = mp_python.BaseOptions(model_asset_path='pose_landmarker_full.task')
        pose_cfg  = mp_vision.PoseLandmarkerOptions(
            base_options=pose_opts,
            running_mode=mp_vision.RunningMode.VIDEO,
            num_poses=1,
            min_pose_detection_confidence=0.5,
            min_pose_presence_confidence=0.5,
            min_tracking_confidence=0.5,
        )
        self.pose_landmarker = mp_vision.PoseLandmarker.create_from_options(pose_cfg)

        hand_opts = mp_python.BaseOptions(model_asset_path='hand_landmarker.task')
        hand_cfg  = mp_vision.HandLandmarkerOptions(
            base_options=hand_opts,
            running_mode=mp_vision.RunningMode.VIDEO,
            num_hands=2,
            min_hand_detection_confidence=0.5,
            min_hand_presence_confidence=0.5,
            min_tracking_confidence=0.5,
        )
        self.hand_landmarker = mp_vision.HandLandmarker.create_from_options(hand_cfg)
        self._ts = 0

    def _lm(self, landmarks, n):
        out = [[0.0, 0.0, 0.0]] * n
        for i, lm in enumerate(landmarks[:n]):
            out[i] = [lm.x, lm.y, lm.z]
        return out

    def extract_frame(self, bgr):
        self._ts += 33
        rgb = cv2.cvtColor(bgr, cv2.COLOR_BGR2RGB)
        img = mp.Image(image_format=mp.ImageFormat.SRGB, data=rgb)
        pr  = self.pose_landmarker.detect_for_video(img, self._ts)
        pose = self._lm(pr.pose_landmarks[0], self.BODY) if pr.pose_landmarks else [[0.,0.,0.]]*self.BODY
        face = [[0.,0.,0.]] * self.FACE
        hr   = self.hand_landmarker.detect_for_video(img, self._ts)
        left = right = [[0.,0.,0.]] * self.HAND
        for i, hl in enumerate(hr.handedness):
            label = hl[0].category_name
            lms   = self._lm(hr.hand_landmarks[i], self.HAND)
            if label == 'Left':  left  = lms
            else:                right = lms
        return np.array(pose + face + left + right, dtype=np.float32)

    def extract_video(self, path, max_frames):
        cap = cv2.VideoCapture(str(path))
        if not cap.isOpened():
            return np.zeros((max_frames, self.TOTAL, 3), dtype=np.float32), 0
        frames = []
        while cap.isOpened() and len(frames) < max_frames:
            ret, fr = cap.read()
            if not ret: break
            frames.append(self.extract_frame(fr))
        cap.release()
        T_real = len(frames)
        if T_real == 0:
            return np.zeros((max_frames, self.TOTAL, 3), dtype=np.float32), 0
        pad = [np.zeros((self.TOTAL, 3), dtype=np.float32)] * (max_frames - T_real)
        return np.stack(frames + pad), T_real

    def close(self):
        self.pose_landmarker.close()
        self.hand_landmarker.close()


KP_EXTRACTOR = KeypointExtractor()
print('KeypointExtractor ready.')


  pose_landmarker_full.task already present.
  hand_landmarker.task already present.
KeypointExtractor ready.


I0000 00:00:1778936489.415423 26903405 gl_context.cc:369] GL version: 2.1 (2.1 Metal - 89.4), renderer: Apple M4 Pro
W0000 00:00:1778936489.458562 26967969 inference_feedback_manager.cc:114] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.
W0000 00:00:1778936489.468181 26967978 inference_feedback_manager.cc:114] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.
I0000 00:00:1778936489.470140 26903405 gl_context.cc:369] GL version: 2.1 (2.1 Metal - 89.4), renderer: Apple M4 Pro
W0000 00:00:1778936489.474836 26967980 inference_feedback_manager.cc:114] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.
W0000 00:00:1778936489.478775 26967984 inference_feedback_manager.cc:114] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.


## Step 6 - Inference Helper Function

In [47]:
RGB_TRANSFORM = T.Compose([
    T.ToPILImage(),
    T.Resize(tuple(CFG['frame_size'])),
    T.ToTensor(),
    T.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])


def load_rgb_frames(video_path, max_frames):
    cap = cv2.VideoCapture(str(video_path))
    frames = []
    while cap.isOpened() and len(frames) < max_frames:
        ret, fr = cap.read()
        if not ret: break
        frames.append(RGB_TRANSFORM(cv2.cvtColor(fr, cv2.COLOR_BGR2RGB)))
    cap.release()
    T_real = len(frames)
    if T_real == 0:
        return torch.zeros(max_frames, 3, *CFG['frame_size']), 0
    pad = [torch.zeros(3, *CFG['frame_size'])] * (max_frames - T_real)
    return torch.stack(frames + pad), T_real


@torch.no_grad()
def predict_video(video_path, confidence_threshold=0.30, top_k=3):
    """
    Returns:
        label      (str | None): predicted word, or None if below threshold
        confidence (float): top-1 softmax score
        top_k_preds (list of (word, score) tuples)
        n_frames   (int): valid frame count extracted
    """
    max_frames = CFG['max_frames']
    rgb, T_real = load_rgb_frames(video_path, max_frames)
    kp,  _      = KP_EXTRACTOR.extract_video(str(video_path), max_frames)

    rgb_t  = rgb.unsqueeze(0).to(DEVICE)
    kp_t   = torch.from_numpy(kp).unsqueeze(0).to(DEVICE)
    mask_t = torch.zeros(1, max_frames, device=DEVICE)
    mask_t[0, :T_real] = 1.0

    logits, _ = model(rgb_t, kp_t, mask_t)
    probs      = F.softmax(logits, dim=-1)[0]

    top_k_vals, top_k_idx = probs.topk(min(top_k, N_CLASSES))
    top_k_preds = [(IDX2CLASS[i.item()], v.item()) for i, v in zip(top_k_idx, top_k_vals)]

    confidence = top_k_preds[0][1]
    label      = top_k_preds[0][0] if confidence >= confidence_threshold else None
    return label, confidence, top_k_preds, T_real


print('Inference function ready.')


Inference function ready.


## Step 7 - Interactive Video Inference UI

Run this cell to launch the upload widget.
1. Click **Upload Video** and select your `.mp4` clip
2. Click **Translate Sign** to run inference
3. The predicted word (or fallback message) appears below with a confidence chart


In [48]:
ACCENT  = '#2563EB'
SUCCESS = '#16A34A'
DANGER  = '#DC2626'
BG_CARD = '#F8FAFF'

title_html = widgets.HTML(value=(
    '<div style="background:linear-gradient(135deg,#2563EB,#1e40af);'
    'color:white;border-radius:12px;padding:18px 24px;margin-bottom:12px;'
    'font-family:Arial,sans-serif;">'
    '<h2 style="margin:0;font-size:1.4rem;">ASL Sign Language Translator</h2>'
    f'<p style="margin:4px 0 0;opacity:0.88;font-size:0.9rem;">'
    f'Dual-Stream RGB + Keypoint Model &middot; {N_CLASSES} signs &middot; '
    f'confidence threshold = {CONFIDENCE_THRESHOLD:.0%}</p>'
    '</div>'
))

upload_btn = widgets.FileUpload(
    accept='.mp4,.avi,.mov', multiple=False,
    description='Upload Video',
    layout=widgets.Layout(width='220px')
)
run_btn = widgets.Button(
    description='Translate Sign',
    button_style='primary',
    layout=widgets.Layout(width='160px', height='36px')
)
clear_btn = widgets.Button(
    description='Clear',
    button_style='warning',
    layout=widgets.Layout(width='90px', height='36px')
)
status_lbl = widgets.HTML(value='<span style="color:gray;">No video uploaded yet.</span>')
result_out = widgets.Output()
controls   = widgets.HBox([upload_btn, run_btn, clear_btn],
                           layout=widgets.Layout(gap='10px', align_items='center'))


def render_result(label, confidence, top_k_preds, n_frames, video_name):
    clear_output(wait=True)
    if label:
        banner_color = SUCCESS
        icon = 'Predicted Sign:'
        main_text = f'<b style="font-size:2rem;">{label.upper()}</b>'
        sub_text  = f'Confidence: {confidence*100:.1f}%  |  {n_frames} frames processed'
    else:
        banner_color = DANGER
        icon = ''
        main_text = '<b style="font-size:1.4rem;">Sorry, I can\'t translate this word.</b>'
        sub_text  = (f'Top prediction "{top_k_preds[0][0]}" scored only '
                     f'{confidence*100:.1f}% (threshold: {CONFIDENCE_THRESHOLD*100:.0f}%)')

    display(HTML(
        f'<div style="background:{BG_CARD};border:2px solid {banner_color};'
        f'border-radius:12px;padding:20px 28px;font-family:Arial,sans-serif;margin-bottom:14px;">'
        f'<div style="color:gray;font-size:0.85rem;">{icon}</div>'
        f'<div style="color:{banner_color};">{main_text}</div>'
        f'<div style="color:#555;font-size:0.88rem;margin-top:6px;">{sub_text}</div>'
        f'<div style="color:#888;font-size:0.8rem;margin-top:4px;">File: {video_name}</div>'
        f'</div>'
    ))

    # Confidence bar chart
    words  = [p[0] for p in top_k_preds]
    scores = [p[1] for p in top_k_preds]
    bar_colors = [
        SUCCESS if (i == 0 and label) else
        (DANGER if (i == 0 and not label) else '#93C5FD')
        for i in range(len(words))
    ]
    fig, ax = plt.subplots(figsize=(7, 2.8))
    bars = ax.barh(words[::-1], scores[::-1], color=bar_colors[::-1],
                   edgecolor='white', height=0.55)
    ax.axvline(CONFIDENCE_THRESHOLD, color='gray', linestyle='--',
               linewidth=1.2, label=f'Threshold ({CONFIDENCE_THRESHOLD:.0%})')
    for bar, score in zip(bars, scores[::-1]):
        ax.text(min(score + 0.01, 0.97), bar.get_y() + bar.get_height() / 2,
                f'{score*100:.1f}%', va='center', fontsize=10, fontweight='bold')
    ax.set_xlim(0, 1.15)
    ax.set_xlabel('Softmax Probability')
    ax.set_title(f'Top-{len(top_k_preds)} Predictions', fontweight='bold')
    ax.legend(loc='lower right', fontsize=9)
    ax.grid(axis='x', alpha=0.3)
    plt.tight_layout()
    plt.show()


def on_run(b):
    with result_out:
        if not upload_btn.value:
            status_lbl.value = '<span style="color:orange;">Please upload a video first.</span>'
            return

        raw = upload_btn.value
        uploaded   = raw[0] if isinstance(raw, tuple) else list(raw.values())[0]
        video_name = uploaded['name']
        video_bytes = uploaded['content']

        tmp = tempfile.NamedTemporaryFile(suffix='.mp4', delete=False)
        tmp.write(video_bytes)
        tmp.flush()
        tmp_path = tmp.name
        tmp.close()

        status_lbl.value = f'<span style="color:#2563EB;">Processing {video_name}...</span>'

        try:
            label, confidence, top_k_preds, n_frames = predict_video(
                tmp_path,
                confidence_threshold=CONFIDENCE_THRESHOLD,
                top_k=TOP_K,
            )
            status_lbl.value = '<span style="color:green;">Done.</span>'
            render_result(label, confidence, top_k_preds, n_frames, video_name)
        except Exception as e:
            status_lbl.value = f'<span style="color:red;">Error: {e}</span>'
        finally:
            os.unlink(tmp_path)


def on_clear(b):
    with result_out:
        clear_output()
    status_lbl.value = '<span style="color:gray;">No video uploaded yet.</span>'


run_btn.on_click(on_run)
clear_btn.on_click(on_clear)

ui = widgets.VBox(
    [title_html, controls, status_lbl, result_out],
    layout=widgets.Layout(padding='12px', max_width='760px')
)
display(ui)
